# Chit 1 baseline: reading out a qubit · **FPGA-ns**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

Train a tiny quantized network on raw I/Q readout traces (800 numbers per shot), measure the readout fidelity on the held-out shots, and check that the hls4ml emulation is bit-exact. It copies the published design's shape: 800 → 4 → 2 with **ternary** first-layer weights.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~5 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

In [ ]:
# --- Colab setup: run once per session (~2 min: it installs Brevitas and hls4ml) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["brevitas", "hls4ml==1.3.0", "quantizers<2", "qonnx", "onnxoptimizer", "onnx"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

## 1. Data
One shot = 400 time samples of I and Q (interleaved). Label 0 = |g⟩, 1 = |e⟩. The test split is the published one.

In [ ]:
from capstone.data.qubit import load, fidelity
N_TRAIN = 50_000 if FAST else 300_000             # a subset of the 800k training shots
X_tr, y_tr = load("train", max_items=N_TRAIN)
X_va, y_va = load("val", max_items=20_000 if FAST else None)
X_te, y_te = load("test")
print("train", X_tr.shape, "| val", X_va.shape, "| test", X_te.shape, "| fraction |e>:", y_te.mean().round(3))

**Input scaling.** The traces are ADC counts. We divide them by one **power of two** (a bit shift on the FPGA) chosen on the training split, so most values
land in [-1, 1] for the 8-bit input quantizer.

In [ ]:
SCALE = 2.0 ** np.ceil(np.log2(np.quantile(np.abs(X_tr), 0.999)))
Xtr, Xva, Xte = X_tr / SCALE, X_va / SCALE, X_te / SCALE
print("scale:", SCALE)

## 2. The model, costed before training
800 → 4 → 2. First layer **2-bit weights** (with `qat_mlp` that is exactly ternary: −1, 0, +1, so its "multiplications" are additions), second layer 6-bit;
all inputs 8-bit.

In [ ]:
from common.compress import qat_mlp
SIZES = (800, 4, 2)
CFG = [(2, 8), (6, 8)]
model = qat_mlp(CFG, SIZES)
p = profile(model, (800,))
print(f"params {p['params']:,} | MACs {p['macs']:,}")

## 3. Train and measure the fidelity

In [ ]:
fit(model, Xtr, y_tr, epochs=2 if FAST else 8, bs=512, lr=3e-3)
F_TEST = fidelity(y_te, predict(model, Xte).argmax(1))
print(f"readout fidelity (test): {F_TEST:.4f}   (val: {fidelity(y_va, predict(model, Xva).argmax(1)):.4f})")

## 4. On the FPGA: bit-exact emulation and what hls4ml builds
Resources and latency in ns come from **C-synthesis** (instructors, `common/submit.py`). The budget is ≤ 50 ns at II = 1: fully unrolled (RF = 1).

In [ ]:
from common.hls import to_hls, check_bit_exact, dense_summary
hm, _ = to_hls(model, output_dir="hls_prj_qubit", in_features=800)
diff, agree = check_bit_exact(hm, model, Xte[:2000])
print(f"max |Brevitas - hls4ml| on 2,000 shots: {diff:.3g}, same decision on {100 * agree:.2f}%")
DS = dense_summary(hm)
for r in DS:
    print(r)
MULTS = sum(r["mults_needed"] for r in DS)

In [ ]:
ROW = report("1 qubit", (len(Xtr), len(Xva), len(Xte)), (800,), f"fidelity {F_TEST:.4f}", p,
             f"FPGA RF=1: {MULTS:,} multipliers (dense_summary); ns need C-synthesis; bit-exact={diff == 0}",
             "CC BY 4.0 data (Zenodo, confirmed); code repo has no licence (only its recipe reused)",
             extra=dict(minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.